# DLinear — Walmart Store Sales Forecasting

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ["WALMART_ROOT"] = "/content/drive/MyDrive/MLFinalAssignment"

Mounted at /content/drive


In [ ]:
import importlib.util
import os
import pathlib
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    missing = [p for p in ("mlflow", "dagshub")
               if importlib.util.find_spec(p) is None]
    if missing:
        print("installing", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if not pathlib.Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


def find_root() -> pathlib.Path:
    """Locate the repo: it must hold train.csv and src/walmart_prep.py."""
    candidates = []
    if os.environ.get("WALMART_ROOT"):
        candidates.append(pathlib.Path(os.environ["WALMART_ROOT"]))
    candidates += [pathlib.Path.cwd(), pathlib.Path.cwd().parent]
    if IN_COLAB:
        drive_root = pathlib.Path("/content/drive/MyDrive")
        candidates += [drive_root / "MLFinalProject", pathlib.Path("/content/MLFinalProject")]
        if drive_root.exists():
            candidates += sorted(p for p in drive_root.glob("*") if (p / "train.csv").exists())
    for c in candidates:
        if (c / "train.csv").exists() and (c / "src" / "walmart_prep.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the project. Set WALMART_ROOT to the folder containing "
        "train.csv and src/walmart_prep.py, then re-run this cell.\n"
        f"Looked in: {[str(c) for c in candidates]}")


ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
for sub in ("docs", "submissions"):
    (ROOT / sub).mkdir(exist_ok=True)

print(f"IN_COLAB={IN_COLAB}\nROOT={ROOT}")

IN_COLAB=True
ROOT=/content/drive/MyDrive/MLFinalAssignment


In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='smama23', repo_name='MLFinalProject', mlflow=True)

EXPERIMENT_NAME = 'DLinear_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

Accessing as smama23

Initialized MLflow to track repo "smama23/MLFinalProject"

Repository smama23/MLFinalProject initialized!

Tracking URI: https://dagshub.com/smama23/MLFinalProject.mlflow
Experiment  : DLinear_Training


In [ ]:
import time

import numpy as np
import pandas as pd

import torch
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline

from walmart_panel import WalmartPanel
from walmart_dl import DLinear, train_dlinear, DLinearForecaster

from walmart_prep import (
    FOLDS, HORIZON, MARKDOWN_COLS, MIN_SAFE_LAG,
    SeasonalResidualRegressor, WalmartFeatureBuilder,
    december_shape_report, load_raw, make_submission,
    score_fold, seasonal_naive, wmae_weights,
)

EXPERIMENT = EXPERIMENT_NAME
print("tracking:", mlflow.get_tracking_uri())

train, test, features, stores = load_raw(str(ROOT))
SEED = 0

print(f"train {train.shape}   test {test.shape}")
print(f"train {train.Date.min().date()} .. {train.Date.max().date()}")
print(f"test  {test.Date.min().date()} .. {test.Date.max().date()}")
print(f"horizon = {HORIZON} weeks  ->  minimum safe lag = {MIN_SAFE_LAG} weeks")

tracking: https://dagshub.com/smama23/MLFinalProject.mlflow
train (421570, 5)   test (115064, 4)
train 2010-02-05 .. 2012-10-26
test  2012-11-02 .. 2013-07-26
horizon = 39 weeks  ->  minimum safe lag = 39 weeks


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}  torch={torch.__version__}")
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

device=cuda  torch=2.11.0+cu128
gpu: Tesla T4


## `DLinear_Cleaning`

In [ ]:
with mlflow.start_run(run_name="DLinear_Cleaning"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_params({
        "panel": "dense [n_series x n_weeks], internal gaps -> 0",
        "normalisation": "per-series z-score from training weeks only",
        "loss": "WMAE-weighted L1 (holiday horizon steps x5), in normalised space",
        "target_modes": "level (raw series) and residual (YoY change over lag-52 baseline)",
    })
    supply = []
    for L in (52, 39):
        for fold in list(FOLDS) + [None]:
            cut = None if fold is None else fold.cut
            name = "FULL" if fold is None else fold.name
            try:
                p = WalmartPanel(train, features, stores, lookback=L).fit(cut)
                n = len(p.make_windows())
                supply.append({"lookback": L, "fold": name, "weeks": p.n_weeks_, "windows": n})
            except ValueError:
                supply.append({"lookback": L, "fold": name, "weeks": None, "windows": 0})
    supply = pd.DataFrame(supply)
    for _, r in supply.iterrows():
        mlflow.log_metric(f"windows_L{r.lookback}_{r.fold}", r.windows)
supply.pivot(index="fold", columns="lookback", values="windows")

🏃 View run DLinear_Cleaning at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/b72ad9b6f73f400984a03dddd9373c38
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2


lookback,39,52
fold,,
FULL,178171,142522
early,2699,0
mirror,37721,2683
recent,72761,37597


## `DLinear_Baseline`

In [ ]:
baseline_scores = {}
with mlflow.start_run(run_name="DLinear_Baseline"):
    mlflow.log_param("model", "seasonal naive: lag-52, fallback pair median, then global median")
    for fold in FOLDS:
        tr = train[train.Date <= fold.cut]
        va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
        s = score_fold(va.Weekly_Sales, seasonal_naive(tr, va), va)
        baseline_scores[fold.name] = s
        mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])

BASELINE = {k: v["wmae"] for k, v in baseline_scores.items()}
pd.DataFrame(baseline_scores).round(1)

🏃 View run DLinear_Baseline at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/29b5d25b5f6b4bba96f8d40a673ffadf
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2


,mirror,recent,early
wmae,2037.8,1807.2,2018.6
mae,1949.9,1805.5,1955.9
mae_holiday,2320.2,1815.7,2170.0
mae_nonholiday,1918.6,1804.9,1931.2
bias,-381.1,-375.3,-341.3
mae_thanksgiving,2387.6,NaN,2399.5
mae_christmas,2711.7,NaN,2721.0
mae_superbowl,1860.7,1832.5,1874.9
mae_laborday,NaN,1798.6,1675.8


## `DLinear_CV`

In [ ]:
EPOCHS = 80
cv_rows = []
with mlflow.start_run(run_name="DLinear_CV"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_params({"epochs": EPOCHS, "lr": 2e-3, "kernel": 25, "target_mode": "level"})
    trials = [("recent", 52), ("recent", 39), ("mirror", 39)]
    for fold_name, L in trials:
        fold = next(f for f in FOLDS if f.name == fold_name)
        va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
        panel = WalmartPanel(train, features, stores, lookback=L, target_mode="level").fit(fold.cut)
        net = train_dlinear(panel, epochs=EPOCHS, lr=2e-3, kernel=25,
                            batch_size=1024, device=DEVICE, seed=0, verbose=False)
        s = score_fold(va.Weekly_Sales, DLinearForecaster(panel, net)(va), va)
        row = {"fold": fold_name, "lookback": L, "wmae": s["wmae"],
               "naive": BASELINE[fold_name], "vs_naive_%": 100 * (1 - s["wmae"] / BASELINE[fold_name])}
        cv_rows.append(row)
        mlflow.log_metric(f"wmae_{fold_name}_L{L}", s["wmae"])
        print(f"level  {fold_name:7s} L={L}: WMAE={s['wmae']:7.1f}  naive={BASELINE[fold_name]:7.1f}  "
              f"({row['vs_naive_%']:+.1f}%)")

pd.DataFrame(cv_rows).round(1)

level  recent  L=52: WMAE= 2393.8  naive= 1807.2  (-32.5%)
level  recent  L=39: WMAE= 2336.5  naive= 1807.2  (-29.3%)
level  mirror  L=39: WMAE= 3313.5  naive= 2037.8  (-62.6%)
🏃 View run DLinear_CV at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/d56890958c404de4b77710bac1eaba32
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2


,fold,lookback,wmae,naive,vs_naive_%
0,recent,52,2393.8,1807.2,-32.5
1,recent,39,2336.5,1807.2,-29.3
2,mirror,39,3313.5,2037.8,-62.6


## `DLinear_ResidualStudy`

In [ ]:
fold = next(f for f in FOLDS if f.name == "recent")
va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
panel_r = WalmartPanel(train, features, stores, lookback=52, target_mode="residual").fit(fold.cut)

zero = DLinear(52, 39, 25)
with torch.no_grad():
    for p in zero.parameters():
        p.zero_()
zero_wmae = score_fold(va.Weekly_Sales, DLinearForecaster(panel_r, zero)(va), va)["wmae"]
print(f"residual DLinear, ZERO net = {zero_wmae:.1f}  (naive = {BASELINE['recent']:.1f})\n")

study = [{"weight_decay": 0.0, "wmae": zero_wmae, "note": "zero net == naive"}]
with mlflow.start_run(run_name="DLinear_ResidualStudy"):
    mlflow.log_param("device", DEVICE)
    mlflow.log_metric("zero_net_wmae", zero_wmae)
    for wd in (1e-4, 1e-2, 1e-1, 1.0, 10.0):
        net = train_dlinear(panel_r, epochs=100, lr=1e-3, weight_decay=wd,
                            batch_size=1024, device=DEVICE, seed=0, verbose=False)
        s = score_fold(va.Weekly_Sales, DLinearForecaster(panel_r, net)(va), va)
        study.append({"weight_decay": wd, "wmae": s["wmae"], "note": ""})
        mlflow.log_metric("wmae_vs_weight_decay", s["wmae"])
        print(f"weight_decay={wd:<7} WMAE={s['wmae']:7.1f}  ({100*(1-s['wmae']/BASELINE['recent']):+.1f}% vs naive)")

pd.DataFrame(study).round(1)

residual DLinear, ZERO net = 1807.2  (naive = 1807.2)

weight_decay=0.0001  WMAE= 2391.7  (-32.3% vs naive)
weight_decay=0.01    WMAE= 2269.6  (-25.6% vs naive)
weight_decay=0.1     WMAE= 1946.8  (-7.7% vs naive)
weight_decay=1.0     WMAE= 1816.5  (-0.5% vs naive)
weight_decay=10.0    WMAE= 1807.8  (-0.0% vs naive)
🏃 View run DLinear_ResidualStudy at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/9258f2a050414bf198151fd62fd2f22e
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2


,weight_decay,wmae,note
0,0.0,1807.2,zero net == naive
1,0.0,2391.7,
2,0.0,2269.6,
3,0.1,1946.8,
4,1.0,1816.5,
5,10.0,1807.8,


## `DLinear_Tuning`

In [ ]:
rng = np.random.default_rng(0)
GRID = {
    "lookback": [39, 52],
    "target_mode": ["level", "residual"],
    "lr": [5e-4, 1e-3, 2e-3],
    "kernel": [13, 25],
    "weight_decay": [1e-2, 1e-1, 1.0],
    "epochs": [80],
}
N_TRIALS = 8
fold = next(f for f in FOLDS if f.name == "recent")
va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]

def sample_cfg():
    return {k: (type(v[0])(rng.choice(v)) if not isinstance(v[0], str) else str(rng.choice(v)))
            for k, v in GRID.items()}

panels = {(L, m): WalmartPanel(train, features, stores, lookback=L, target_mode=m).fit(fold.cut)
          for L in GRID["lookback"] for m in GRID["target_mode"]}

trials = []
with mlflow.start_run(run_name="DLinear_Tuning"):
    mlflow.log_params({"n_trials": N_TRIALS, "selection_fold": "recent", "device": DEVICE})
    for i in range(N_TRIALS):
        cfg = sample_cfg()
        panel = panels[(cfg["lookback"], cfg["target_mode"])]
        net = train_dlinear(panel, epochs=cfg["epochs"], lr=cfg["lr"], kernel=cfg["kernel"],
                            weight_decay=cfg["weight_decay"], batch_size=1024, device=DEVICE,
                            seed=0, verbose=False)
        s = score_fold(va.Weekly_Sales, DLinearForecaster(panel, net)(va), va)
        cfg["wmae"] = s["wmae"]
        trials.append(cfg)
        with mlflow.start_run(run_name=f"DLinear_Tuning__trial_{i:02d}", nested=True):
            mlflow.log_params(cfg); mlflow.log_metric("wmae_recent", s["wmae"])
        print(f"trial {i:02d}  WMAE={s['wmae']:7.1f}  {cfg['target_mode']:8s} L={cfg['lookback']} "
              f"lr={cfg['lr']} k={cfg['kernel']} wd={cfg['weight_decay']}")

    best = min(trials, key=lambda t: t["wmae"])
    mlflow.log_params({f"best_{k}": v for k, v in best.items() if k != "wmae"})
    mlflow.log_metric("best_wmae_recent", best["wmae"])

print(f"\nbest: WMAE={best['wmae']:.1f}  naive={BASELINE['recent']:.1f}  cfg={best}")
best

🏃 View run DLinear_Tuning__trial_00 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/08b518f040f84539999f5dad1201053b
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2
trial 00  WMAE= 2274.6  residual L=52 lr=0.001 k=13 wd=0.01
🏃 View run DLinear_Tuning__trial_01 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/6a529c9b76f343b5a50e38c8bdbbb1ce
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2
trial 01  WMAE= 2540.8  level    L=39 lr=0.0005 k=13 wd=1.0
🏃 View run DLinear_Tuning__trial_02 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/a0cc633d34e741098d2d31844e33ec3f
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2
trial 02  WMAE= 1816.2  residual L=52 lr=0.001 k=25 wd=1.0
🏃 View run DLinear_Tuning__trial_03 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/5d4ef2c76ecf47

{'lookback': 52,
 'target_mode': 'residual',
 'lr': 0.001,
 'kernel': 25,
 'weight_decay': 1.0,
 'epochs': 80,
 'wmae': 1816.2210651985572}

## `DLinear_Final`

In [ ]:
best_L = int(best["lookback"]); best_mode = best["target_mode"]
panel_full = WalmartPanel(train, features, stores, lookback=best_L, target_mode=best_mode).fit()
net_full = train_dlinear(panel_full, epochs=int(best["epochs"]), lr=float(best["lr"]),
                         kernel=int(best["kernel"]), weight_decay=float(best["weight_decay"]),
                         batch_size=1024, device=DEVICE, seed=0, verbose=True)
forecaster = DLinearForecaster(panel_full, net_full)

with mlflow.start_run(run_name="DLinear_Final") as final_run:
    mlflow.log_param("device", DEVICE)
    mlflow.log_params({k: v for k, v in best.items()})
    mlflow.log_metric("cv_wmae_recent", best["wmae"])

    y_pred = forecaster(test)
    print(f"predictions: n={len(y_pred)} mean={y_pred.mean():.1f} "
          f"min={y_pred.min():.1f} max={y_pred.max():.1f}")

    from walmart_prep import WalmartFeatureBuilder
    g = (WalmartFeatureBuilder(features, stores)
         .fit(train.drop(columns=["Weekly_Sales"]), train.Weekly_Sales).xmas_profile_)
    frame, verdict = december_shape_report(train, test, y_pred, g)
    print(frame.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
    print(f"\npredicted peak {verdict['peak_predicted']} | implied {verdict['peak_implied']}")
    print("GATE:", "PASS" if verdict["passed"] else "REVIEW -- " + "; ".join(verdict["problems"]))
    mlflow.log_metric("december_xmas_gap_pct", verdict["xmas_gap_pct"])
    mlflow.log_metric("december_gate_passed", int(verdict["passed"]))

    logged_model = mlflow.pyfunc.log_model(
        name="model", python_model=forecaster,
        code_paths=[str(ROOT / "src" / "walmart_prep.py"),
                    str(ROOT / "src" / "walmart_panel.py"),
                    str(ROOT / "src" / "walmart_dl.py")],
        input_example=test.head(3),
    )
    MODEL_URI = logged_model.model_uri

    sub = make_submission(test, y_pred, ROOT / "submissions" / "dlinear.csv")
    mlflow.log_artifact(str(ROOT / "submissions" / "dlinear.csv"))
    FINAL_RUN_ID = final_run.info.run_id

print(f"\nrun_id = {FINAL_RUN_ID}\nmodel_uri = {MODEL_URI}")
sub.head()

  epoch   0  weighted L1 (norm) = 0.7053
  epoch  10  weighted L1 (norm) = 0.6441
  epoch  20  weighted L1 (norm) = 0.6441
  epoch  30  weighted L1 (norm) = 0.6442
  epoch  40  weighted L1 (norm) = 0.6442
  epoch  50  weighted L1 (norm) = 0.6442
  epoch  60  weighted L1 (norm) = 0.6442
  epoch  70  weighted L1 (norm) = 0.6442
  epoch  79  weighted L1 (norm) = 0.6442
predictions: n=115064 mean=16458.4 min=-498.0 max=646111.5
      week  k_end  predicted  implied  gap_pct
2012-11-30    -25      0.984    0.969    1.513
2012-12-07    -18      1.114    1.109    0.387
2012-12-14    -11      1.217    1.208    0.749
2012-12-21     -4      1.574    1.489    5.691
2012-12-28      3      0.944    1.243  -24.007
2013-01-04     10      0.932    0.919    1.428

predicted peak 2012-12-21 | implied 2012-12-21
GATE: REVIEW -- Christmas week off by -24.0%, beyond the profile's uncertainty


2026/07/10 18:13:27 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/10 18:13:27 INFO mlflow.pyfunc: Inferring model signature from input example
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64)

🏃 View run DLinear_Final at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2/runs/71032c1260304913bcb9884c92b21ae1
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/2

run_id = 71032c1260304913bcb9884c92b21ae1
model_uri = models:/m-e3386cb91e9b4a2a818dedac19f87fbf


,Id,Weekly_Sales
0,1_1_2012-11-02,39342.440362
1,1_1_2012-11-09,18386.112030
2,1_1_2012-11-16,18529.131764
3,1_1_2012-11-23,20501.915829
4,1_1_2012-11-30,24746.653469


In [ ]:
from mlflow import MlflowClient
loaded = mlflow.pyfunc.load_model(MODEL_URI)
reloaded = np.asarray(loaded.predict(test))
print("max |reloaded - original| =", float(np.abs(reloaded - y_pred).max()))
assert np.allclose(reloaded, y_pred, atol=1e-3), "the logged pyfunc must reproduce its predictions"
print("pyfunc round-trip OK")

mv = mlflow.register_model(MODEL_URI, REGISTERED_MODEL_NAME)
print(f"registered {mv.name} version {mv.version}")
try:
    MlflowClient().set_registered_model_alias(mv.name, "dlinear", mv.version)
    print(f"alias 'dlinear' -> version {mv.version}")
except Exception as e:
    print("alias not set (optional):", str(e)[:80])

max |reloaded - original| = 0.0
pyfunc round-trip OK


Registered model 'WalmartSalesForecast' already exists. Creating a new version of this model...
2026/07/10 18:14:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: WalmartSalesForecast, version 4
Created version '4' of model 'WalmartSalesForecast'.


registered WalmartSalesForecast version 4
alias 'dlinear' -> version 4


In [ ]:
print("experiments: https://dagshub.com/smama23/MLFinalProject/experiments")
print("registry   : https://dagshub.com/smama23/MLFinalProject/models")
try:
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
    cols = [c for c in ("tags.mlflow.runName", "metrics.wmae_recent_L52",
                        "metrics.december_gate_passed") if c in runs.columns]
    summary = runs[cols] if cols else runs
    try:
        display(summary)
    except NameError:
        print(summary.to_string(index=False))
except Exception as e:
    print("could not fetch run summary (model is already logged):", e)

experiments: https://dagshub.com/smama23/MLFinalProject/experiments
registry   : https://dagshub.com/smama23/MLFinalProject/models


,tags.mlflow.runName,metrics.wmae_recent_L52,metrics.december_gate_passed
0,DLinear_Final,NaN,0.0
1,DLinear_Tuning__trial_07,NaN,NaN
2,DLinear_Tuning__trial_06,NaN,NaN
3,DLinear_Tuning__trial_05,NaN,NaN
4,DLinear_Tuning__trial_04,NaN,NaN
5,DLinear_Tuning__trial_03,NaN,NaN
6,DLinear_Tuning__trial_02,NaN,NaN
7,DLinear_Tuning__trial_01,NaN,NaN
8,DLinear_Tuning__trial_00,NaN,NaN
9,DLinear_Tuning,NaN,NaN
